## Uppgift 2 — Sportstatistik

Välj 2-4 sporter och skapa lämpliga grafer/diagram för att visualisera exempelvis:
- medaljfördelning mellan länder i sporterna
- åldersfördelning i sporterna

Skapa fler plots för att visualisera olika aspekter kring sporterna.

In [1]:
import pandas as pd
import hashlib

# Read csv

olympics = pd.read_csv("./data/athlete_events.csv")

olympics["Name"] = olympics["Name"].apply(
    lambda x: hashlib.sha256(x.encode()).hexdigest()
)

In [2]:
from normalize_noc import normalize_noc

# Normalise NOC codes

noc = pd.read_csv("./data/noc_regions.csv")

clean = normalize_noc(olympics, noc)

event_medals = clean[clean["Medal"].notna()].drop_duplicates(
    subset=["Games", "Event", "NOC_folded", "Medal"]
)

⚠️ Unmapped NOCs (add to FOLD_MAP or noc_regions): ['ROT', 'TUV', 'UNK']


### Fencing

In [3]:
import plotly.express as px

fencing_medals = (
    event_medals[event_medals["Sport"] == "Fencing"]
    .groupby("NOC_folded")["Medal"]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig_fencing = px.bar(
    fencing_medals,
    x="NOC_folded",
    y="Medal",
    title="Top Countries in Fencing (Event-Level Medals)",
)

fig_fencing.update_layout(
    xaxis_title="Country",
    yaxis_title="Medal count"
)
fig_fencing.show()

### Athletics

In [4]:
athletics_medals = (
    event_medals[event_medals["Sport"] == "Athletics"]
    .groupby("NOC_folded")["Medal"]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig_ath = px.bar(
    athletics_medals,
    x="NOC_folded",
    y="Medal",
    title="Top Countries in Athletics (Event-Level Medals)",
)

fig_ath.update_layout(
    xaxis_title="Country",
    yaxis_title="Medal count"
)


fig_ath.show()

In [5]:
# Long distance events

long_distance_events = [
    "Athletics Women's Marathon",
    "Athletics Men's Marathon",
    "Athletics Men's 3,000 metres Steeplechase",
    "Athletics Women's 3,000 metres Steeplechase",
    "Athletics Women's 5,000 metres",
    "Athletics Men's 5,000 metres",
    "Athletics Men's 10,000 metres",
    "Athletics Women's 10,000 metres",
    "Athletics Women's Marathon",
    "Athletics Men's Marathon"
]

ld_medals = event_medals[
    (event_medals["Sport"] == "Athletics") &
    (event_medals["Event"].isin(long_distance_events))
]

ld_medal_counts = (
    ld_medals
    .groupby("NOC_folded")["Medal"]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig_ld = px.bar(
    ld_medal_counts,
    x="NOC_folded",
    y="Medal",
    title="Top Countries in Long-Distance Athletics",
)

fig_ld.update_layout(
    xaxis_title="Country",
    yaxis_title="Medal count"
)

fig_ld.show()


In [6]:
## Timeline of East African dominance

east_africa = (
    ld_medals[ld_medals["NOC"].isin(["KEN", "ETH"])]
    .groupby(["Year", "NOC", "Games"])["Medal"]
    .count()
    .reset_index()
)

east_africa

px.line(
    east_africa,
    x="Year",
    y="Medal",
    color="NOC",
    title="Rise of Kenya and Ethiopia in Long-Distance Running",
).show()

# # Note to self: 1976 and 1980 were boycott years

### Cross-country skiing

In [7]:
xc_ski_medals = (
    event_medals[event_medals["Sport"] == "Cross Country Skiing"]
    .groupby("NOC_folded")["Medal"]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig_ski = px.bar(
    xc_ski_medals,
    x="NOC_folded",
    y="Medal",
    title="Top Countries in Cross Country Skiing (Event-Level Medals)",
)
fig_ski.update_layout(
    xaxis_title="Country",
    yaxis_title="Medal count"
)
fig_ski.show()


### Medals by country over the years

In [8]:
yearly_medals = event_medals.groupby(["Year", "NOC"]).size().reset_index(name="Count")

yearly_medals = yearly_medals.merge(
    noc[["NOC", "region"]], on="NOC", how="left"
)

yearly_medals = yearly_medals.dropna(subset=["region"])

fig = px.choropleth(
    yearly_medals,
    locations="region",
    locationmode="country names",
    color="Count",
    hover_name="region",
    animation_frame="Year",
    color_continuous_scale="Plasma",
    title="Olympic medals over time",
)
fig.show()


/var/folders/z6/lqrq2m_5737_gdx8z2b2c2v00000gn/T/ipykernel_46418/3028256959.py:9: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.

